# Drive `outputs/` 정리 스크립트

학습 중간에 저장된 epoch별 체크포인트 (`*.pt`, `project_z_*.npy`, `company_z_*.npy`, `metrics_*_epoch*.json`) 중 **peak / final 외 모든 중간 파일**을 Drive에서 삭제합니다.

## 동작 방식

- **KEEP**: peak + final epoch만 (~139 파일, ~20 GB)
- **DELETE**: 중간 epoch 파일 (~1027 파일, ~106 GB)  → Drive 휴지통으로 이동 (**30일 복구 가능**)
- **UNTOUCHED**: 알 수 없는 tag → 자동 보호 (예: 다른 계정에서 합친 MPW 결과)

## 안전장치

1. `--dry-run` 또는 `--execute` 둘 중 하나 명시 필수 — 실수로 삭제 안 되게
2. KEEP 리스트에 없는 tag는 자동 보존
3. Drive 휴지통 30일 보관 → 잘못 삭제 시 복구 가능

## 사용 순서

1. **Mount + Pull repo** (셀 1, 2)
2. **현재 상태 확인** (셀 3)
3. **DRY-RUN으로 계획 미리보기** (셀 4) ← 반드시 먼저 실행
4. 결과 OK면 **실제 삭제** (셀 5)
5. **결과 확인** (셀 6)

## Step 0. Drive mount + 저장소 동기화

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_DIR = '/content/apollo.M1.GNN'
GIT_URL = 'https://github.com/osung/apollo.M1.GNN.git'

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git pull origin main
else:
    !git clone $GIT_URL $REPO_DIR
    %cd $REPO_DIR

## Step 1. 현재 outputs/ 상태 확인

정리 대상 폴더가 존재하는지, 몇 개의 파일이 있는지 확인합니다.

**다른 계정/폴더에서 사용하려면** 아래 `OUTPUTS_DIR`을 수정하세요.

In [ ]:
OUTPUTS_DIR = '/content/drive/MyDrive/apollo.M1.GNN/outputs'

import os, subprocess
if not os.path.isdir(OUTPUTS_DIR):
    print(f'[WARN] {OUTPUTS_DIR} 가 존재하지 않습니다. 경로 확인 필요.')
else:
    n = len(os.listdir(OUTPUTS_DIR))
    sz = subprocess.check_output(['du', '-sh', OUTPUTS_DIR], text=True).split()[0]
    print(f'경로: {OUTPUTS_DIR}')
    print(f'파일 수: {n}')
    print(f'총 크기: {sz}')

## Step 2. DRY-RUN — 계획 미리보기

실제로 아무 파일도 삭제하지 않고 **무엇을 KEEP / DELETE / UNTOUCHED로 분류했는지**만 출력합니다. 반드시 먼저 실행해서 의도와 일치하는지 검토하세요.

In [ ]:
!python scripts/cleanup_drive_outputs.py --outputs-dir "$OUTPUTS_DIR" --dry-run

## Step 3. 실제 삭제 (`--execute`)

위 dry-run 결과가 의도와 맞으면 아래 셀을 실행하세요. 삭제된 파일은 **Drive 휴지통으로 이동**되어 30일간 복구 가능합니다.

**주의**: 셀 실행 전 변경된 KEEP 리스트가 필요하면 `scripts/cleanup_drive_outputs.py`의 `KEEP` dict를 직접 수정 후 git push → 다시 pull.

In [ ]:
!python scripts/cleanup_drive_outputs.py --outputs-dir "$OUTPUTS_DIR" --execute

## Step 4. 결과 확인

삭제 후 폴더 상태를 다시 확인합니다.

In [ ]:
import subprocess
n = len(os.listdir(OUTPUTS_DIR))
sz = subprocess.check_output(['du', '-sh', OUTPUTS_DIR], text=True).split()[0]
print(f'정리 후 파일 수: {n}')
print(f'정리 후 크기: {sz}')

# 남아있는 tag 목록
import re
from collections import Counter
rx = re.compile(r'^model_(.+)_epoch(\d+)\.pt$')
tags = Counter()
for f in os.listdir(OUTPUTS_DIR):
    m = rx.match(f)
    if m:
        tags[m.group(1)] += 1
print('\n남아있는 model 체크포인트 tag별 epoch 수:')
for tag, n in sorted(tags.items()):
    print(f'  {tag}: {n}')

## (선택) 다른 폴더 정리

예를 들어 `sweep/` 폴더도 정리하고 싶다면:

```python
OUTPUTS_DIR = '/content/drive/MyDrive/apollo.M1.GNN/sweep/gfm'
```

으로 바꿔서 Step 1 ~ 4를 다시 실행. KEEP 리스트는 `outputs/`에 맞춰져 있어서 sweep 폴더 정리는 어울리지 않을 수 있습니다 (sweep 파일명 체계가 다름).